# Noise Map — Inference only (saved model from train_noise_model.py)

**All training is in `train_noise_model.py`.** This notebook is for **inference only**: load the saved model and use it.

1. **Load data** + **EDA** — explore noisecapture_prepared.parquet (optional; needed if you want to evaluate on the same test set).
2. **Load model** — load checkpoint and preprocess from `train_noise_model.py` output (`CKPT_DIR`).
3. **Evaluate** — test-set metrics and predicted vs actual (optional).
4. **Heatmap demo** — user-selected region and time → predicted noise (dB) map.

**Prerequisite:** Run `python train_noise_model.py --data ... --out-dir checkpoints` first. Then set `CKPT_DIR` in the notebook (e.g. `checkpoints/`).

## 1. Setup & Load data from Google Drive

Data file: [noisecapture_prepared.parquet](https://drive.google.com/file/d/1iQx7NaDg1GkN7ZbQLeNyMmFIHczsNUfT/view?usp=drive_link) (prepared by `data_prep/prep_data.py`).

In [ ]:
# Run in Colab: mount Drive and install deps if needed
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

if IN_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  !pip install -q pyarrow pandas matplotlib seaborn scikit-learn torch
else:
  %pip install -q pyarrow pandas matplotlib seaborn scikit-learn torch

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Data path: use Drive file ID for direct download in Colab, or local path
FILE_ID = '1iQx7NaDg1GkN7ZbQLeNyMmFIHczsNUfT'
PARQUET_NAME = 'noisecapture_prepared.parquet'

def load_data():
  if IN_COLAB:
    # Option A: file already in Drive (e.g. in My Drive/NoiseMap/)
    drive_path = Path('/content/drive/My Drive')
    for candidate in [drive_path / 'NoiseMap' / PARQUET_NAME,
                      drive_path / PARQUET_NAME,
                      Path('/content') / PARQUET_NAME]:
      if candidate.exists():
        return pd.read_parquet(candidate)
    # Option B: download with gdown
    !pip install -q gdown
    import gdown
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    out = '/content/noisecapture_prepared.parquet'
    gdown.download(url, out, quiet=False, fuzzy=True)
    return pd.read_parquet(out)
  else:
    # Local: try common locations
    for p in [Path('data_prep') / PARQUET_NAME, Path(PARQUET_NAME)]:
      if p.exists():
        return pd.read_parquet(p)
    raise FileNotFoundError('Parquet not found. In Colab, add file to Drive or use gdown.')

df = load_data()
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Drop rows with missing AEF_embed (required for ML)
df_clean = df.dropna(subset=['AEF_embed']).copy()
if 'geometry' in df_clean.columns:
  df_clean = df_clean.drop(columns=['geometry'])
print('Rows with valid AEF_embed:', len(df_clean))
df_clean.head()

In [ ]:
# Scatter: sample locations around the world (subsample for speed)
np.random.seed(42)
n_plot = min(50_000, len(df_clean))
sub = df_clean.sample(n_plot)
fig, ax = plt.subplots(figsize=(14, 6))
sc = ax.scatter(sub['lon'], sub['lat'], c=sub['noise_level_dB'], s=0.3, alpha=0.5, cmap='viridis')
plt.colorbar(sc, label='Noise (dB)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('NoiseCapture samples (world)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Time / hour distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_clean['hour'].hist(bins=24, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Hour of day (UTC)')
axes[0].set_ylabel('Count')
axes[0].set_title('Samples by hour of day')
axes[0].set_xticks(range(0, 25, 2))
# Noise level by hour (binned)
df_clean.assign(hour_bin=(df_clean['hour'] // 1).astype(int)).boxplot(
  column='noise_level_dB', by='hour_bin', ax=axes[1], grid=False)
axes[1].set_xlabel('Hour of day (UTC)')
axes[1].set_ylabel('Noise (dB)')
axes[1].set_title('Noise level by hour')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 countries by sample count
top10 = df_clean['country'].value_counts().head(10)
fig, ax = plt.subplots(figsize=(10, 5))
top10.plot(kind='barh', ax=ax)
ax.set_xlabel('Number of samples')
ax.set_ylabel('Country')
ax.set_title('Top 10 countries by sample count')
plt.tight_layout()
plt.show()

In [ ]:
# Noise (dB) distribution and extra views
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
df_clean['noise_level_dB'].hist(ax=axes[0,0], bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Noise (dB)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Distribution of noise level (dB)')
# Top 10: mean noise by country
by_country = df_clean.groupby('country')['noise_level_dB'].agg(['mean','count'])
by_country = by_country[by_country['count'] >= 100].nlargest(10, 'count')
by_country['mean'].plot(kind='barh', ax=axes[0,1])
axes[0,1].set_xlabel('Mean noise (dB)')
axes[0,1].set_title('Mean noise (dB) — top 10 countries by count')
# Day vs night
if 'is_night' in df_clean.columns:
  sns.boxplot(data=df_clean, x='is_night', y='noise_level_dB', ax=axes[1,0])
  axes[1,0].set_xticklabels(['Day', 'Night'])
  axes[1,0].set_xlabel('Is night (22h–6h)')
  axes[1,0].set_ylabel('Noise (dB)')
  axes[1,0].set_title('Noise: day vs night')
# Lat/lon density
axes[1,1].hexbin(df_clean['lon'], df_clean['lat'], gridsize=40, cmap='Blues', mincnt=1)
axes[1,1].set_xlabel('Longitude')
axes[1,1].set_ylabel('Latitude')
axes[1,1].set_title('Sample density (hexbin)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 3. Load model and preprocess (from train_noise_model.py)

Load the checkpoint and preprocess saved by `train_noise_model.py`. Set `CKPT_DIR` to the folder that contains `noise_mlp_checkpoint.pt` and `noise_mlp_preprocess.joblib`.

In [ ]:
import joblib
from torch import nn

# Where train_noise_model.py wrote the checkpoint (e.g. checkpoints/ or .)
CKPT_DIR = Path('checkpoints')
if IN_COLAB:
  CKPT_DIR = Path('/content/drive/My Drive/NoiseMap')
ckpt_path = CKPT_DIR / 'noise_mlp_checkpoint.pt'
preprocess_path = CKPT_DIR / 'noise_mlp_preprocess.joblib'
if not ckpt_path.exists():
  raise FileNotFoundError(f'Run train_noise_model.py first. Missing: {ckpt_path}')
if not preprocess_path.exists():
  raise FileNotFoundError(f'Missing: {preprocess_path}')

ckpt = torch.load(ckpt_path, map_location=device)
config = ckpt['config']
n_features = ckpt['n_features']

class NoiseMLP(nn.Module):
  def __init__(self, n_in, hidden=(128, 64)):
    super().__init__()
    layers = []
    prev = n_in
    for h in hidden:
      layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.BatchNorm1d(h)]
      prev = h
    layers += [nn.Linear(prev, 1)]
    self.net = nn.Sequential(*layers)
  def forward(self, x):
    return self.net(x).squeeze(-1)

model = NoiseMLP(n_features, tuple(config['hidden'])).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

preprocess = joblib.load(preprocess_path)
scaler = preprocess['scaler']
ohe = preprocess['ohe']
le_country = preprocess['le_country']
country_classes = preprocess['country_classes']
aef_cols = preprocess['aef_cols']
feature_cols = preprocess['feature_cols']

print('Model and preprocess loaded from', CKPT_DIR)

## 4. Prepare test data and evaluate

Build features from `df_clean` using the **loaded** scaler and encoders (same split as `train_noise_model.py`: test_size=0.2, random_state=42), then compute metrics and predicted vs actual plot.

In [ ]:
from sklearn.model_selection import train_test_split

# Build features (same as train_noise_model.py): AEF + lat, lon, hour_sin, hour_cos + country one-hot
if not all(c in df_clean.columns for c in aef_cols):
  aef = np.stack(df_clean['AEF_embed'].values)
  for i, c in enumerate(aef_cols):
    df_clean[c] = aef[:, i]
df_clean['country_idx'] = le_country.transform(df_clean['country'].astype(str))
X_num = df_clean[feature_cols].values
country_onehot = ohe.transform(df_clean[['country_idx']])
X = np.hstack([X_num, country_onehot]).astype(np.float32)
y = df_clean['noise_level_dB'].values.astype(np.float32)

# Same split as script (seed=42, stratify by country)
try:
  _, test_idx = train_test_split(
    np.arange(len(df_clean)), test_size=0.2, random_state=42,
    stratify=df_clean['country_idx']
  )
except ValueError:
  _, test_idx = train_test_split(np.arange(len(df_clean)), test_size=0.2, random_state=42)
X_test = X[test_idx]
y_test = y[test_idx]
X_test_s = scaler.transform(X_test).astype(np.float32)

print('Test size:', len(y_test))

In [ ]:
# Training is done by train_noise_model.py — this notebook only runs inference (evaluate + heatmap below).

In [ ]:
# Evaluate: metrics and predicted vs actual
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()
with torch.no_grad():
  y_pred = model(torch.from_numpy(X_test_s).to(device)).cpu().numpy()
print('Test — MAE: {:.2f} dB, RMSE: {:.2f} dB, R²: {:.4f}'.format(
  mean_absolute_error(y_test, y_pred),
  np.sqrt(mean_squared_error(y_test, y_pred)),
  r2_score(y_test, y_pred)))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=5)
ax.plot([30, 110], [30, 110], 'k--', label='Perfect')
ax.set_xlabel('Actual noise (dB)')
ax.set_ylabel('Predicted noise (dB)')
ax.set_title('PyTorch MLP: predicted vs actual')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Heatmap demo — predicted noise (dB) for a region and time

Select **bbox** (west, south, east, north) and **hour** (0–24). Loads AEF for that region (`aef_loader` + `odc-geo`), runs the loaded model, shows heatmap with colorbar. Requires: `pip install aef-loader odc-geo pyproj nest_asyncio`

In [ ]:
# User-selected region and time
bbox = [2.2, 48.8, 2.5, 49.0]   # [west, south, east, north] — e.g. Paris area
hour = 14.0                      # 0–24
resolution = 100                 # meters per pixel (smaller = finer, slower)
MAX_PIXELS = 300_000             # subsample if grid is larger

def run_heatmap_demo(bbox, hour, resolution=100, max_pixels=MAX_PIXELS):
  try:
    import asyncio
    import nest_asyncio
    nest_asyncio.apply()
    from pyproj import Transformer
    from odc.geo.geobox import GeoBox
    from aef_loader import AEFIndex, VirtualTiffReader, DataSource
    from aef_loader.utils import dequantize_aef, reproject_datatree
  except ImportError as e:
    print('Install aef-loader and odc-geo for the heatmap demo:')
    print('  pip install aef-loader odc-geo pyproj nest_asyncio')
    return

  target_crs = 'EPSG:3857'
  year = 2024
  transformer = Transformer.from_crs('EPSG:4326', target_crs, always_xy=True)
  inv_transformer = Transformer.from_crs(target_crs, 'EPSG:4326', always_xy=True)
  x_min, y_min = transformer.transform(bbox[0], bbox[1])
  x_max, y_max = transformer.transform(bbox[2], bbox[3])
  geobox = GeoBox.from_bbox(bbox=(x_min, y_min, x_max, y_max), crs=target_crs, resolution=resolution)

  async def load_aef():
    index = AEFIndex(source=DataSource.SOURCE_COOP)
    await index.download()
    tiles = await index.query(bbox=bbox, years=(year, year))
    async with VirtualTiffReader() as reader:
      tree = await reader.open_tiles_by_zone(tiles)
    combined = reproject_datatree(tree, target_geobox=geobox)
    emb = combined.embeddings.isel(time=0)
    if hasattr(emb, 'data') and hasattr(emb.data, 'compute'):
      emb = emb.compute()
    raw = dequantize_aef(emb).values
    return np.moveaxis(raw, 0, -1)

  loop = asyncio.get_event_loop()
  aef_grid = loop.run_until_complete(load_aef())
  H, W = aef_grid.shape[0], aef_grid.shape[1]
  n_full = H * W
  if n_full > max_pixels:
    stride = max(1, int((n_full / max_pixels) ** 0.5))
    aef_grid = aef_grid[::stride, ::stride, :].copy()
    H, W = aef_grid.shape[0], aef_grid.shape[1]
  aef_flat = aef_grid.reshape(-1, 64)

  # Pixel centers -> lon, lat
  transform = geobox.transform
  lons, lats = [], []
  for i in range(H):
    for j in range(W):
      x_proj, y_proj = transform * (j + 0.5, i + 0.5)
      lon, lat = inv_transformer.transform(x_proj, y_proj)
      lons.append(lon)
      lats.append(lat)
  lons = np.array(lons).reshape(H, W)
  lats = np.array(lats).reshape(H, W)

  hour_sin = np.sin(2 * np.pi * hour / 24)
  hour_cos = np.cos(2 * np.pi * hour / 24)
  default_country_idx = 0
  country_onehot = np.zeros((H * W, len(country_classes)), dtype=np.float32)
  country_onehot[:, default_country_idx] = 1.0

  X_demo = np.hstack([
    aef_flat,
    lats.ravel()[:, None], lons.ravel()[:, None],
    np.full((H * W, 1), hour_sin), np.full((H * W, 1), hour_cos),
    country_onehot
  ]).astype(np.float32)
  X_demo_s = scaler.transform(X_demo)

  model.eval()
  pred_demo = []
  batch_size = 8192
  with torch.no_grad():
    for i in range(0, len(X_demo_s), batch_size):
      xb = torch.from_numpy(X_demo_s[i:i+batch_size]).to(device)
      pred_demo.append(model(xb).cpu().numpy())
  pred_demo = np.concatenate(pred_demo).reshape(H, W)

  extent = [bbox[0], bbox[2], bbox[1], bbox[3]]  # west, east, south, north
  fig, ax = plt.subplots(figsize=(10, 8))
  im = ax.imshow(pred_demo, extent=extent, origin='upper', aspect='auto',
                 cmap='viridis', vmin=45, vmax=85)
  ax.set_xlabel('Longitude')
  ax.set_ylabel('Latitude')
  ax.set_title(f'Predicted noise (dB) — hour={hour}')
  plt.colorbar(im, ax=ax, label='Noise (dB)')
  plt.tight_layout()
  plt.show()

run_heatmap_demo(bbox, hour, resolution, MAX_PIXELS)

In [ ]:
# Optional: if heatmap fails with ImportError, run once:
# !pip install aef-loader odc-geo pyproj nest_asyncio

In [ ]:
# Optional: install aef-loader for heatmap demo (run once if the heatmap cell failed with ImportError)
# !pip install aef-loader odc-geo pyproj nest_asyncio